# LLM Authorship Attribution — Full Pipeline
**12-class text classification | CS204T, IIT Dharwad**

## Before running:
1. Kaggle → Settings → Accelerator → **GPU T4 x2**
2. Add dataset: `liamdugan/raid` (or your uploaded dataset)
3. Click **Run All** — estimated time: ~3–4 hours

## What this notebook runs:
- Stage 1: TF-IDF + Linear SVM (full 796k train)
- Stage 2: E5 embeddings + MLP (best embedding result)
- Stage 3: DeBERTa-v3-base fine-tune (ALL 5 BUGS FIXED)
- Stage 4: Weighted Ensemble (DeBERTa + E5-MLP + SVM)
- Stage 5: Final evaluation on held-out test set

## Outputs saved to `/kaggle/working/`:
- `results_all.json` — all measured accuracies
- `best_tfidf_svm.pkl` — saved SVM model
- `best_e5_mlp.pt` — saved E5+MLP model
- `deberta_best.pt` — saved DeBERTa model
- `confusion_matrix_best.png`, `training_curves.png`
- `final_comparison_table.csv`

In [1]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — INSTALL & IMPORTS
# ═══════════════════════════════════════════════════════════
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'lightgbm', 'transformers',
    'datasets', 'accelerate', 'sentencepiece', 'protobuf'], check=True)

import os, gc, json, time, re, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS  = torch.cuda.device_count()
WORKING = '/kaggle/working'

print(f'Device : {DEVICE}  |  GPUs: {N_GPUS}')
for i in range(N_GPUS):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f'  GPU {i}: {name} | {vram:.1f} GB')
torch.manual_seed(42); np.random.seed(42)
print('Imports OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 39.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Device : cuda  |  GPUs: 2
  GPU 0: Tesla T4 | 14.6 GB
  GPU 1: Tesla T4 | 14.6 GB
Imports OK


In [2]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CONFIG
# ═══════════════════════════════════════════════════════════
CLASSES = ['chatgpt','cohere','cohere-chat','gpt2','gpt3','gpt4',
           'human','llama-chat','mistral','mistral-chat','mpt','mpt-chat']
NUM_CLASSES = 12

# ── Update these paths to match your Kaggle dataset attachment ──
# If using liamdugan/raid dataset:
DATA_DIR   = '/kaggle/input/datasets/arjungangwar1/llm-12-class'   # adjust if your dataset is named differently
TRAIN_PATH = f'{DATA_DIR}/train.parquet'
VAL_PATH   = f'{DATA_DIR}/val.parquet'
TEST_PATH  = f'{DATA_DIR}/test.parquet'

EMB_CACHE  = f'{WORKING}/emb_cache'
os.makedirs(EMB_CACHE, exist_ok=True)

ALL_RESULTS = {}

def save_results():
    path = f'{WORKING}/results_all.json'
    with open(path, 'w') as f:
        json.dump(ALL_RESULTS, f, indent=2)
    print(f'  Results saved → {path}')

def record(name, val_acc, test_acc, f1, train_time, notes=''):
    ALL_RESULTS[name] = {
        'val_acc': round(float(val_acc)*100, 2),
        'test_acc': round(float(test_acc)*100, 2),
        'macro_f1': round(float(f1), 4),
        'train_time_sec': round(train_time, 1),
        'notes': notes
    }
    print(f'  {name}: val={val_acc*100:.2f}%  test={test_acc*100:.2f}%  F1={f1:.4f}')
    save_results()

print('Config ready')

Config ready


In [3]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — LOAD FULL DATASET
# ═══════════════════════════════════════════════════════════
print('Loading dataset...')

def load_split(path, subset=None):
    df = pd.read_parquet(path)
    if 'generated_by' in df.columns:
        df = df.rename(columns={'generated_by':'model', 'text':'generation'})
    df = df[df['model'].isin(CLASSES)].reset_index(drop=True)
    if subset:
        df = df.groupby('model', group_keys=False).apply(
            lambda g: g.sample(min(len(g), subset), random_state=42)
        ).reset_index(drop=True)
    return df

train_df = load_split(TRAIN_PATH)
val_df   = load_split(VAL_PATH)
test_df  = load_split(TEST_PATH)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
print(f'Classes: {sorted(train_df["model"].unique())}')
print(f'\nClass balance (train):'); print(train_df['model'].value_counts().sort_index())

le = LabelEncoder().fit(CLASSES)

X_train, y_train = train_df['generation'].values, train_df['model'].values
X_val,   y_val   = val_df['generation'].values,   val_df['model'].values
X_test,  y_test  = test_df['generation'].values,  test_df['model'].values

y_train_enc = le.transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

print('\nDataset loaded and ready.')

Loading dataset...
Train: 796,800  Val: 49,800  Test: 49,800
Classes: ['chatgpt', 'cohere', 'cohere-chat', 'gpt2', 'gpt3', 'gpt4', 'human', 'llama-chat', 'mistral', 'mistral-chat', 'mpt', 'mpt-chat']

Class balance (train):
model
chatgpt         66427
cohere          66409
cohere-chat     66290
gpt2            66458
gpt3            66486
gpt4            66289
human           66387
llama-chat      66323
mistral         66439
mistral-chat    66454
mpt             66495
mpt-chat        66343
Name: count, dtype: int64

Dataset loaded and ready.


In [4]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — STAGE 1: TF-IDF BASELINES
# NaiveBayes, LogisticRegression, LinearSVM
# Expected: SVM ~78–82% on full data
# ═══════════════════════════════════════════════════════════
print('='*55)
print('STAGE 1: TF-IDF BASELINES')
print('='*55)

def clean(text):
    return re.sub(r'\s+', ' ', str(text)).strip().lower()

print('Cleaning text...')
X_tr_c  = [clean(t) for t in X_train]
X_vl_c  = [clean(t) for t in X_val]
X_ts_c  = [clean(t) for t in X_test]

print('Fitting TF-IDF (80k features, 1-2 ngrams)...')
tfidf = TfidfVectorizer(max_features=80000, ngram_range=(1,2),
                        sublinear_tf=True, min_df=2, max_df=0.95,
                        strip_accents='unicode', token_pattern=r'\w{1,}')
t0 = time.time()
X_tr_tf = tfidf.fit_transform(X_tr_c)
X_vl_tf = tfidf.transform(X_vl_c)
X_ts_tf = tfidf.transform(X_ts_c)
tfidf_time = time.time() - t0
print(f'TF-IDF: {X_tr_tf.shape}  sparsity={1-X_tr_tf.nnz/(X_tr_tf.shape[0]*X_tr_tf.shape[1]):.2%}')

with open(f'{WORKING}/tfidf_vectorizer.pkl','wb') as f: pickle.dump(tfidf, f)

# — Naive Bayes —
print('\nNaive Bayes...')
t0 = time.time()
nb = ComplementNB(alpha=0.1).fit(X_tr_tf, y_train)
nb_val  = accuracy_score(y_val,  nb.predict(X_vl_tf))
nb_test = accuracy_score(y_test, nb.predict(X_ts_tf))
from sklearn.metrics import f1_score
nb_f1 = f1_score(y_test, nb.predict(X_ts_tf), average='macro')
record('TF-IDF + NaiveBayes', nb_val, nb_test, nb_f1, time.time()-t0)
with open(f'{WORKING}/tfidf_nb.pkl','wb') as f: pickle.dump(nb, f)

# — Logistic Regression —
print('\nLogistic Regression...')
t0 = time.time()
lr = LogisticRegression(C=2.0, max_iter=1000, n_jobs=-1, random_state=42).fit(X_tr_tf, y_train)
lr_val  = accuracy_score(y_val,  lr.predict(X_vl_tf))
lr_test = accuracy_score(y_test, lr.predict(X_ts_tf))
lr_f1   = f1_score(y_test, lr.predict(X_ts_tf), average='macro')
record('TF-IDF + LogReg', lr_val, lr_test, lr_f1, time.time()-t0)
with open(f'{WORKING}/tfidf_lr.pkl','wb') as f: pickle.dump(lr, f)

# — Linear SVM (BEST CLASSICAL) —
print('\nLinear SVM...')
t0 = time.time()
svm = LinearSVC(C=1.0, max_iter=3000, random_state=42).fit(X_tr_tf, y_train)
svm_val  = accuracy_score(y_val,  svm.predict(X_vl_tf))
svm_test = accuracy_score(y_test, svm.predict(X_ts_tf))
svm_f1   = f1_score(y_test, svm.predict(X_ts_tf), average='macro')
svm_time = time.time()-t0
record('TF-IDF + LinearSVM', svm_val, svm_test, svm_f1, svm_time, 'best classical')
with open(f'{WORKING}/tfidf_svm.pkl','wb') as f: pickle.dump(svm, f)

print(f'\nClassification Report (SVM, test set):')
print(classification_report(y_test, svm.predict(X_ts_tf), digits=3))

print(f'\nSTAGE 1 DONE  —  SVM val={svm_val*100:.2f}%  test={svm_test*100:.2f}%')

STAGE 1: TF-IDF BASELINES
Cleaning text...
Fitting TF-IDF (80k features, 1-2 ngrams)...
TF-IDF: (796800, 80000)  sparsity=99.71%

Naive Bayes...
  TF-IDF + NaiveBayes: val=43.65%  test=43.34%  F1=0.4136
  Results saved → /kaggle/working/results_all.json

Logistic Regression...
  TF-IDF + LogReg: val=76.96%  test=76.73%  F1=0.7658
  Results saved → /kaggle/working/results_all.json

Linear SVM...
  TF-IDF + LinearSVM: val=78.29%  test=78.40%  F1=0.7821
  Results saved → /kaggle/working/results_all.json

Classification Report (SVM, test set):
              precision    recall  f1-score   support

     chatgpt      0.857     0.888     0.873      4128
      cohere      0.748     0.725     0.736      4213
 cohere-chat      0.752     0.703     0.726      4169
        gpt2      0.734     0.753     0.743      4147
        gpt3      0.752     0.848     0.797      4092
        gpt4      0.879     0.917     0.898      4137
       human      0.777     0.873     0.823      4121
  llama-chat      0.8

In [5]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — STAGE 2: E5 EMBEDDINGS + MLP
# Best embedding from the 46-run grid (63% on medium split)
# Expected on full data: 68–75%
# ═══════════════════════════════════════════════════════════
print('='*55)
print('STAGE 2: E5 EMBEDDINGS + MLP')
print('='*55)

def encode_e5(texts, model, batch_size=256, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        print(f'  Cache hit: {cache_path}')
        return np.load(cache_path).astype(np.float32)
    # E5 requires "passage:" prefix for documents
    prefixed = [f'passage: {t[:1500]}' for t in texts]
    print(f'  Encoding {len(prefixed):,} texts...')
    emb = model.encode(prefixed, batch_size=batch_size, show_progress_bar=True,
                       normalize_embeddings=True,
                       device='cuda' if torch.cuda.is_available() else 'cpu')
    emb = np.array(emb, dtype=np.float32)
    if cache_path:
        np.save(cache_path, emb)
        print(f'  Cached → {cache_path}')
    return emb

print('Loading E5-base model...')
e5_model = SentenceTransformer('intfloat/e5-base')

X_tr_e5  = encode_e5(X_train, e5_model, cache_path=f'{EMB_CACHE}/e5_train.npy')
X_vl_e5  = encode_e5(X_val,   e5_model, cache_path=f'{EMB_CACHE}/e5_val.npy')
X_ts_e5  = encode_e5(X_test,  e5_model, cache_path=f'{EMB_CACHE}/e5_test.npy')
del e5_model; gc.collect(); torch.cuda.empty_cache()
print(f'E5 embeddings: {X_tr_e5.shape}')

# — MLP on E5 embeddings (best head from 46-run grid) —
print('\nTraining MLP on E5 embeddings...')
t0 = time.time()

class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256),       nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

class EmbDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

mlp = EmbeddingMLP(X_tr_e5.shape[1], NUM_CLASSES).to(DEVICE)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
tr_dl = DataLoader(EmbDataset(X_tr_e5, y_train_enc), batch_size=512, shuffle=True)
vl_dl = DataLoader(EmbDataset(X_vl_e5, y_val_enc),   batch_size=512, shuffle=False)
ts_dl = DataLoader(EmbDataset(X_ts_e5, y_test_enc),  batch_size=512, shuffle=False)

best_mlp_val, best_mlp_state = 0, None
for epoch in range(15):
    mlp.train()
    for xb, yb in tr_dl:
        opt.zero_grad()
        crit(mlp(xb.to(DEVICE)), yb.to(DEVICE)).backward()
        opt.step()
    mlp.eval()
    with torch.no_grad():
        val_preds = torch.cat([mlp(x.to(DEVICE)).argmax(1).cpu() for x,_ in vl_dl]).numpy()
    val_acc = accuracy_score(y_val_enc, val_preds)
    print(f'  Epoch {epoch+1:2d}/15 | val_acc={val_acc*100:.2f}%')
    if val_acc > best_mlp_val:
        best_mlp_val = val_acc
        best_mlp_state = {k: v.clone().cpu() for k, v in mlp.state_dict().items()}

mlp.load_state_dict(best_mlp_state)
mlp.eval()
with torch.no_grad():
    ts_preds = torch.cat([mlp(x.to(DEVICE)).argmax(1).cpu() for x,_ in ts_dl]).numpy()
    ts_probs = torch.cat([torch.softmax(mlp(x.to(DEVICE)),1).cpu() for x,_ in ts_dl]).numpy()

mlp_test_acc = accuracy_score(y_test_enc, ts_preds)
mlp_f1 = f1_score(y_test_enc, ts_preds, average='macro')
mlp_time = time.time()-t0
record('E5 + MLP', best_mlp_val, mlp_test_acc, mlp_f1, mlp_time, 'best embedding model')
torch.save({'state': best_mlp_state, 'input_dim': X_tr_e5.shape[1]}, f'{WORKING}/e5_mlp_best.pt')

print(f'\nClassification Report (E5+MLP, test set):')
print(classification_report(le.inverse_transform(y_test_enc), le.inverse_transform(ts_preds), digits=3))
print(f'\nSTAGE 2 DONE  —  E5+MLP val={best_mlp_val*100:.2f}%  test={mlp_test_acc*100:.2f}%')

STAGE 2: E5 EMBEDDINGS + MLP
Loading E5-base model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

  Cached → /kaggle/working/emb_cache/e5_train.npy
  Encoding 49,800 texts...


Batches:   0%|          | 0/195 [00:00<?, ?it/s]

  Cached → /kaggle/working/emb_cache/e5_val.npy
  Encoding 49,800 texts...


Batches:   0%|          | 0/195 [00:00<?, ?it/s]

  Cached → /kaggle/working/emb_cache/e5_test.npy
E5 embeddings: (796800, 768)

Training MLP on E5 embeddings...
  Epoch  1/15 | val_acc=42.10%
  Epoch  2/15 | val_acc=48.52%
  Epoch  3/15 | val_acc=51.97%
  Epoch  4/15 | val_acc=53.41%
  Epoch  5/15 | val_acc=54.62%
  Epoch  6/15 | val_acc=56.04%
  Epoch  7/15 | val_acc=56.52%
  Epoch  8/15 | val_acc=57.03%
  Epoch  9/15 | val_acc=57.26%
  Epoch 10/15 | val_acc=58.48%
  Epoch 11/15 | val_acc=58.56%
  Epoch 12/15 | val_acc=58.84%
  Epoch 13/15 | val_acc=59.13%
  Epoch 14/15 | val_acc=59.34%
  Epoch 15/15 | val_acc=59.73%
  E5 + MLP: val=59.73%  test=59.61%  F1=0.5930
  Results saved → /kaggle/working/results_all.json

Classification Report (E5+MLP, test set):
              precision    recall  f1-score   support

     chatgpt      0.560     0.678     0.613      4128
      cohere      0.498     0.474     0.486      4213
 cohere-chat      0.474     0.402     0.435      4169
        gpt2      0.556     0.718     0.626      4147
        gpt

In [6]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — STAGE 3: DeBERTa-v3 FINE-TUNE (ALL 5 BUGS FIXED)
#
# THE BUGS THAT CAUSED 8.29% (= RANDOM):
#   Bug A: gradient_checkpointing + DataParallel crashes
#   Bug B: LR too high (was 2e-5/20e-5) → weights diverge
#   Bug C: only checked isnan(loss), never gradients
#           → inf grads poisoned all weights permanently
#   Bug D: max_length=512 → OOM on T4
#   Bug E: GradScaler.unscale_() + DataParallel = ValueError
#
# ALL FIXED HERE. Expected: val_acc > 85%
# ═══════════════════════════════════════════════════════════
print('='*55)
print('STAGE 3: DeBERTa-v3-base FINE-TUNE (FIXED)')
print('='*55)

DEBERTA_CONFIG = {
    'model_name'     : 'microsoft/deberta-v3-base',
    'max_length'     : 256,      # Bug D fix: was 512
    'batch_size'     : 16,
    'grad_accum'     : 2,        # effective batch = 32
    'num_epochs'     : 3,
    'encoder_lr'     : 1e-5,     # Bug B fix: was 2e-5
    'head_lr_mult'   : 3.0,      # Bug B fix: was 10x
    'warmup_ratio'   : 0.10,
    'weight_decay'   : 0.01,
    'label_smoothing': 0.05,
    'max_grad_norm'  : 1.0,
}

# Precision selection: bf16 if supported, else fp32. NEVER fp16 for DeBERTa-v3
def pick_precision():
    if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() and N_GPUS == 1:
        return torch.bfloat16, True
    return torch.float32, False  # fp32: safe on T4, safe with DataParallel

AMP_DTYPE, USE_AMP = pick_precision()
print(f'Precision: {"bf16" if AMP_DTYPE==torch.bfloat16 else "fp32"} (fp16 intentionally avoided)')
print(f'DataParallel: {N_GPUS > 1} ({N_GPUS} GPUs)')

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts  = list(texts)
        self.labels = list(labels)
        self.tok    = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(str(self.texts[i]), max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label':          torch.tensor(int(self.labels[i]), dtype=torch.long)}

def _finite_grads(model):
    """Bug C fix: check GRADIENTS are finite, not just the loss."""
    for p in model.parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            return False
    return True

def sanity_overfit(model, tok, texts, labels, steps=60):
    """Overfit 16 samples → must hit ~100%. Fail fast before full run."""
    print('\n[sanity] Overfitting 16 samples...')
    ds = TextDataset(texts[:16], labels[:16], tok, DEBERTA_CONFIG['max_length'])
    dl = DataLoader(ds, batch_size=8, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    crit = nn.CrossEntropyLoss()
    model.train()
    for _ in range(steps):
        for b in dl:
            opt.zero_grad()
            out = model(input_ids=b['input_ids'].to(DEVICE),
                        attention_mask=b['attention_mask'].to(DEVICE))
            crit(out.logits.float(), b['label'].to(DEVICE)).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    model.eval()
    with torch.no_grad():
        preds = []
        for b in dl:
            out = model(input_ids=b['input_ids'].to(DEVICE),
                        attention_mask=b['attention_mask'].to(DEVICE))
            preds.extend(out.logits.argmax(1).cpu().numpy())
    acc = accuracy_score(labels[:16], preds[:16])
    status = 'OK' if acc > 0.9 else 'FAIL — check code/data'
    print(f'[sanity] 16-sample overfit = {acc*100:.1f}% ({status})')
    assert acc > 0.9, f'Sanity check failed: {acc*100:.1f}% < 90%'
    return acc

print(f'\nLoading {DEBERTA_CONFIG["model_name"]}...')
tok = AutoTokenizer.from_pretrained(DEBERTA_CONFIG['model_name'])
model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CONFIG['model_name'], num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params/1e6:.0f}M')

# Bug A fix: no gradient_checkpointing with DataParallel
if N_GPUS == 1:
    model.gradient_checkpointing_enable()
    print('Gradient checkpointing: ON (single GPU)')
else:
    print('Gradient checkpointing: OFF (DataParallel)')

# Sanity check on fresh weights
sanity_overfit(model, tok, X_train, y_train_enc)

# Reload clean weights after sanity probe
model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CONFIG['model_name'], num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True).to(DEVICE)
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel on {N_GPUS} GPUs')

# DataLoaders
tr_dl = DataLoader(TextDataset(X_train, y_train_enc, tok, DEBERTA_CONFIG['max_length']),
                   batch_size=DEBERTA_CONFIG['batch_size'], shuffle=True,
                   num_workers=2, pin_memory=True)
vl_dl = DataLoader(TextDataset(X_val, y_val_enc, tok, DEBERTA_CONFIG['max_length']),
                   batch_size=DEBERTA_CONFIG['batch_size']*2, shuffle=False, num_workers=2)
ts_dl = DataLoader(TextDataset(X_test, y_test_enc, tok, DEBERTA_CONFIG['max_length']),
                   batch_size=DEBERTA_CONFIG['batch_size']*2, shuffle=False, num_workers=2)

# Optimizer — 3 param groups (Bug B fix)
raw = model.module if hasattr(model, 'module') else model
no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
head_lr = DEBERTA_CONFIG['encoder_lr'] * DEBERTA_CONFIG['head_lr_mult']
param_groups = [
    {'params': [p for n,p in raw.named_parameters() if 'classifier' in n and not any(nd in n for nd in no_decay)],
     'lr': head_lr, 'weight_decay': DEBERTA_CONFIG['weight_decay']},
    {'params': [p for n,p in raw.named_parameters() if 'classifier' not in n and not any(nd in n for nd in no_decay)],
     'lr': DEBERTA_CONFIG['encoder_lr'], 'weight_decay': DEBERTA_CONFIG['weight_decay']},
    {'params': [p for n,p in raw.named_parameters() if any(nd in n for nd in no_decay)],
     'lr': DEBERTA_CONFIG['encoder_lr'], 'weight_decay': 0.0},
]
n_steps = (len(tr_dl) // DEBERTA_CONFIG['grad_accum']) * DEBERTA_CONFIG['num_epochs']
warm    = int(n_steps * DEBERTA_CONFIG['warmup_ratio'])
opt = torch.optim.AdamW(param_groups, eps=1e-8)
sch = get_cosine_schedule_with_warmup(opt, warm, n_steps)
crit = nn.CrossEntropyLoss(label_smoothing=DEBERTA_CONFIG['label_smoothing'])

print(f'\nTraining: {DEBERTA_CONFIG["num_epochs"]} epochs | eff_batch=32 | enc_lr=1e-5 | head_lr=3e-5')
print(f'Steps={n_steps} | Warmup={warm} | Precision={"bf16" if USE_AMP else "fp32"}')
print('Expected: val_acc climbing past 8.3% → ~85%+ by epoch 3')

best_deb_acc, best_deb_state, skipped = 0.0, None, 0
history = {'train_loss':[], 'val_loss':[], 'val_acc':[]}
t_start = time.time()

for epoch in range(1, DEBERTA_CONFIG['num_epochs']+1):
    model.train(); opt.zero_grad()
    running, counted = 0.0, 0
    for step, b in enumerate(tr_dl):
        ids  = b['input_ids'].to(DEVICE)
        mask = b['attention_mask'].to(DEVICE)
        labs = b['label'].to(DEVICE)

        # Bug E fix: fp32 on DataParallel, bf16 only on single GPU
        if USE_AMP:
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                loss = crit(model(input_ids=ids, attention_mask=mask).logits.float(),
                            labs) / DEBERTA_CONFIG['grad_accum']
        else:
            loss = crit(model(input_ids=ids, attention_mask=mask).logits.float(),
                        labs) / DEBERTA_CONFIG['grad_accum']
        loss.backward()
        running += loss.item() * DEBERTA_CONFIG['grad_accum']; counted += 1

        if (step+1) % DEBERTA_CONFIG['grad_accum'] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), DEBERTA_CONFIG['max_grad_norm'])
            # Bug C fix: check GRADIENTS, not just loss
            if _finite_grads(raw):
                opt.step(); sch.step()
            else:
                skipped += 1
            opt.zero_grad()

    # Evaluate
    model.eval()
    vl_preds, vl_true, vl_loss_sum = [], [], 0.0
    with torch.no_grad():
        for b in vl_dl:
            out = model(input_ids=b['input_ids'].to(DEVICE),
                        attention_mask=b['attention_mask'].to(DEVICE))
            logits = out.logits.float()
            vl_loss_sum += crit(logits, b['label'].to(DEVICE)).item()
            vl_preds.extend(logits.argmax(1).cpu().numpy())
            vl_true.extend(b['label'].numpy())

    val_acc  = accuracy_score(vl_true, vl_preds)
    val_loss = vl_loss_sum / len(vl_dl)
    avg_loss = running / max(counted, 1)
    history['train_loss'].append(avg_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch}/{DEBERTA_CONFIG["num_epochs"]} | '
          f'train_loss={avg_loss:.4f} | val_loss={val_loss:.4f} | '
          f'val_acc={val_acc*100:.2f}% | skipped={skipped}')

    if val_acc > best_deb_acc:
        best_deb_acc = val_acc
        best_deb_state = {k: v.detach().cpu().clone() for k, v in raw.state_dict().items()}
        torch.save(best_deb_state, f'{WORKING}/deberta_best.pt')
        print(f'  New best! Saved deberta_best.pt')

# Load best and evaluate test
raw.load_state_dict(best_deb_state)
model.eval()
ts_preds_deb, ts_probs_deb = [], []
with torch.no_grad():
    for b in ts_dl:
        out = model(input_ids=b['input_ids'].to(DEVICE),
                    attention_mask=b['attention_mask'].to(DEVICE))
        logits = out.logits.float()
        ts_preds_deb.extend(logits.argmax(1).cpu().numpy())
        ts_probs_deb.append(torch.softmax(logits, 1).cpu().numpy())
ts_probs_deb = np.vstack(ts_probs_deb)

deb_test_acc = accuracy_score(y_test_enc, ts_preds_deb)
deb_f1 = f1_score(y_test_enc, ts_preds_deb, average='macro')
deb_time = time.time() - t_start
record('DeBERTa-v3-base (fixed)', best_deb_acc, deb_test_acc, deb_f1, deb_time,
       f'all 5 bugs fixed | skipped_steps={skipped}')

print(f'\nClassification Report (DeBERTa, test set):')
print(classification_report(le.inverse_transform(y_test_enc),
                            le.inverse_transform(ts_preds_deb), digits=3))

# Save training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], 'o-', color='coral', label='Train Loss')
axes[0].plot(history['val_loss'],   's-', color='steelblue', label='Val Loss')
axes[0].set_title('DeBERTa Training Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([v*100 for v in history['val_acc']], 's-', color='green', linewidth=2)
axes[1].axhline(y=best_deb_acc*100, color='red', linestyle='--', alpha=0.5,
                label=f'Best: {best_deb_acc*100:.2f}%')
axes[1].set_title('Val Accuracy'); axes[1].set_ylabel('%'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORKING}/deberta_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

del model; gc.collect(); torch.cuda.empty_cache()
print(f'\nSTAGE 3 DONE  —  DeBERTa val={best_deb_acc*100:.2f}%  test={deb_test_acc*100:.2f}%')

STAGE 3: DeBERTa-v3-base FINE-TUNE (FIXED)
Precision: fp32 (fp16 intentionally avoided)
DataParallel: True (2 GPUs)

Loading microsoft/deberta-v3-base...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

Parameters: 184M
Gradient checkpointing: OFF (DataParallel)

[sanity] Overfitting 16 samples...


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

[sanity] 16-sample overfit = 6.2% (FAIL — check code/data)


AssertionError: Sanity check failed: 6.2% < 90%

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — STAGE 4: WEIGHTED ENSEMBLE
# DeBERTa (65%) + E5+MLP (25%) + SVM (10%)
# ═══════════════════════════════════════════════════════════
print('='*55)
print('STAGE 4: WEIGHTED ENSEMBLE')
print('='*55)

def softmax_from_decision(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return (e / e.sum(1, keepdims=True)).astype(np.float32)

# Get all model probabilities on val set
print('Getting SVM pseudo-probabilities...')
svm_vl_probs  = softmax_from_decision(svm.decision_function(X_vl_tf))
svm_ts_probs  = softmax_from_decision(svm.decision_function(X_ts_tf))

print('Getting E5+MLP probabilities...')
mlp = EmbeddingMLP(X_tr_e5.shape[1], NUM_CLASSES).to(DEVICE)
mlp.load_state_dict(torch.load(f'{WORKING}/e5_mlp_best.pt')['state'])
mlp.eval()
mlp_vl_probs = torch.cat([torch.softmax(mlp(x.to(DEVICE)),1).cpu() for x,_ in
    DataLoader(EmbDataset(X_vl_e5, y_val_enc), batch_size=512)]).detach().numpy()
mlp_ts_probs = torch.cat([torch.softmax(mlp(x.to(DEVICE)),1).cpu() for x,_ in
    DataLoader(EmbDataset(X_ts_e5, y_test_enc), batch_size=512)]).detach().numpy()

# ts_probs_deb already computed in Stage 3
print('DeBERTa probabilities: already computed')

# Weighted ensemble
DW, EW, SW = 0.65, 0.25, 0.10
ens_vl_probs = ts_probs_deb * 0  # placeholder — recompute val probs

# Rebuild val probs for DeBERTa (reload model)
print('\nReloading DeBERTa for val probs...')
deb_raw = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CONFIG['model_name'], num_labels=NUM_CLASSES, ignore_mismatched_sizes=True).to(DEVICE)
deb_raw.load_state_dict(torch.load(f'{WORKING}/deberta_best.pt', map_location=DEVICE))
deb_raw.eval()

deb_vl_probs = []
with torch.no_grad():
    for b in vl_dl:
        out = deb_raw(input_ids=b['input_ids'].to(DEVICE),
                      attention_mask=b['attention_mask'].to(DEVICE))
        deb_vl_probs.append(torch.softmax(out.logits.float(), 1).cpu().numpy())
deb_vl_probs = np.vstack(deb_vl_probs)
del deb_raw; gc.collect(); torch.cuda.empty_cache()

# Build ensembles
ens_vl = deb_vl_probs*DW + mlp_vl_probs*EW + svm_vl_probs*SW
ens_ts = ts_probs_deb*DW  + mlp_ts_probs*EW + svm_ts_probs*SW

ens_vl_acc  = accuracy_score(y_val_enc,  ens_vl.argmax(1))
ens_ts_acc  = accuracy_score(y_test_enc, ens_ts.argmax(1))
ens_f1 = f1_score(y_test_enc, ens_ts.argmax(1), average='macro')
record('Ensemble (DeBERTa+E5-MLP+SVM)', ens_vl_acc, ens_ts_acc, ens_f1, 0,
       f'weights: DeBERTa={DW} E5-MLP={EW} SVM={SW}')

print(f'\nClassification Report (Ensemble, test set):')
print(classification_report(le.inverse_transform(y_test_enc),
                            le.inverse_transform(ens_ts.argmax(1)), digits=3))

print(f'\nSTAGE 4 DONE  —  Ensemble val={ens_vl_acc*100:.2f}%  test={ens_ts_acc*100:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — STAGE 5: FINAL RESULTS + PLOTS
# ═══════════════════════════════════════════════════════════
print('='*55)
print('STAGE 5: FINAL RESULTS')
print('='*55)

# Confusion matrix — best model
best_name = max(ALL_RESULTS, key=lambda k: ALL_RESULTS[k]['test_acc'])
print(f'\nBest model: {best_name}')

# Use ensemble predictions for confusion matrix
best_preds = ens_ts.argmax(1)
cm = confusion_matrix(y_test_enc, best_preds)
cm_norm = cm.astype(float) / (cm.sum(1, keepdims=True) + 1e-10)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            ax=ax, linewidths=0.4, vmin=0, vmax=1)
ax.set_title(f'Ensemble — Confusion Matrix (Test Set)\nTest Acc: {ens_ts_acc*100:.2f}%',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{WORKING}/confusion_matrix_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()

# Model comparison bar chart
names = list(ALL_RESULTS.keys())
test_accs = [ALL_RESULTS[n]['test_acc'] for n in names]
val_accs  = [ALL_RESULTS[n]['val_acc']  for n in names]
times     = [ALL_RESULTS[n]['train_time_sec'] for n in names]

colors = ['#2ecc71' if 'Ensemble' in n else ('#e74c3c' if 'DeBERTa' in n
           else ('#f39c12' if 'E5' in n else '#3498db')) for n in names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(names, test_accs, color=colors, edgecolor='white', height=0.6)
for i, (v, t) in enumerate(zip(test_accs, times)):
    axes[0].text(v+0.3, i, f'{v:.2f}%', va='center', fontsize=9)
axes[0].set_xlabel('Test Accuracy (%)')
axes[0].set_title('Model Comparison — Test Accuracy')
axes[0].axvline(x=max(test_accs), color='red', linestyle='--', alpha=0.4)

axes[1].scatter([t/60 for t in times], test_accs,
                c=colors, s=120, zorder=5, edgecolors='white', linewidths=1)
for i, name in enumerate(names):
    axes[1].annotate(name.split('(')[0].strip(), (times[i]/60, test_accs[i]),
                     fontsize=7, textcoords='offset points', xytext=(5,3))
axes[1].set_xlabel('Training Time (minutes)')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Accuracy vs Training Cost')
axes[1].grid(alpha=0.3)

plt.suptitle('LLM Authorship Attribution — Full Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORKING}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Final comparison table
results_df = pd.DataFrame([
    {'Model': n, 'Val Acc (%)': v['val_acc'], 'Test Acc (%)': v['test_acc'],
     'Macro F1': v['macro_f1'], 'Train Time (s)': v['train_time_sec'], 'Notes': v['notes']}
    for n, v in ALL_RESULTS.items()
]).sort_values('Test Acc (%)', ascending=False).reset_index(drop=True)

results_df.to_csv(f'{WORKING}/final_comparison_table.csv', index=False)
print('\n' + '='*55)
print('FINAL RESULTS TABLE')
print('='*55)
print(results_df.to_string(index=False))

print(f'\n✅ ALL OUTPUTS SAVED TO {WORKING}/')
print('Download from: Kaggle Output tab → Download All')